In [ ]:
# This tool is used to calculate headway and print transit routs
# Ann Arbor Area Transit GTFS data is at https://www.theride.org/business/software-developers
# Author: cliu / SEMCOG
# Last update: 7/31/2026
# asim1_3

In [ ]:
import os
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import LineString
from datetime import timedelta
import matplotlib.pyplot as plt
import matplotlib.backends.backend_pdf as pdf_backend
import re
_join = os.path.join

In [ ]:
GTFS_location = r'C:\cliu\GitHub\gtfs-headway-explorer'

## Transit Authority GTFS

In [ ]:
sub_path = r'GTFSSample' # input 1
auth = 'AATA' # input 2 - string, e.g., 'AATA', 'DDOT', 'SMART', 'BWAT', 'UMI', 'LET'

stop_times_df = pd.read_csv(_join(GTFS_location, sub_path,'stop_times.txt'))
trips_df = pd.read_csv(_join(GTFS_location, sub_path,'trips.txt'))
calendar_df = pd.read_csv(_join(GTFS_location, sub_path,'calendar.txt'))

index_df = pd.read_csv(_join(GTFS_location, 'All Street Shape File\model_gtfs_index.csv'))
index_df = index_df[index_df['operator'] == auth] 

In [ ]:
# Read service_id from calendar_df
thursday_df = calendar_df[calendar_df['thursday'] == 1].copy()
if not thursday_df.empty:
    service_id = thursday_df.sort_values(by='start_date', ascending=False).service_id.iloc[0]
    # Optional: Update start/end date variables based on this specific record
    row = thursday_df.sort_values(by='start_date', ascending=False).iloc[0]
    start_date = row['start_date']
    end_date = row['end_date']
    service_date = str(start_date) + '-' + str(end_date)
else:
    service_id = None
    print("No service found for Thursday.")

In [ ]:
# Optional: fixed service_id, don't execute otherwise
service_id = 'Jan_to_May_2025-Weekday'
start_date = '20250101'
end_date = '20250531'
service_date = str(start_date) + '-' + str(end_date)

In [ ]:
# Calculate route frequency, ignore NT for E8
model = 'ABM' # input 3
if model == 'E8':
    time_dic = {'AM':['06:30:00', '08:59:59'],'MD':['09:00:00', '14:29:59'],'PM':['14:30:00', '18:29:59'],\
                'EV':['18:30:00', '21:59:59'], 'NT1':['22:00:00', '30:29:59'], 'NT2':['0:00:00', '6:29:59']}  # E8
    duration = {'AM': timedelta(hours=2, minutes=30),  
            'MD': timedelta(hours=5, minutes=30),
            'PM': timedelta(hours=4),
            'EV': timedelta(hours=3, minutes=30),
            'NT': timedelta(hours=8, minutes=30)}
elif model == 'ABM':
    time_dic = {'AM':['06:30:00', '08:59:59'],'MD':['09:00:00', '14:29:59'],'PM':['14:30:00', '18:29:59'],\
                'EV1':['18:30:00', '26:59:59'], 'EV2':['0:00:00', '2:59:59'], 'NT1':['3:00:00', '06:29:59'], 'NT2':['27:00:00', '30:29:59']} # ABM
    duration = {'AM': timedelta(hours=2, minutes=30), 
            'MD': timedelta(hours=5, minutes=30),
            'PM': timedelta(hours=4),
            'EV': timedelta(hours=8, minutes=30), # This is EV in ABM
            'NT': timedelta(hours=3, minutes=30)} # This is EA in ABM

merged_df0 = stop_times_df.merge(trips_df, on='trip_id')
merged_df = merged_df0[merged_df0['service_id'] == service_id].copy()  # Add .copy() to avoid SettingWithCopyWarning

# only keep same stop_id with earlist arrival time   
merged_df['arrival_time'] = pd.to_timedelta(merged_df['arrival_time'])
earliest_indices = merged_df.groupby('shape_id')['arrival_time'].idxmin()
earliest_rows = merged_df.loc[earliest_indices]  
merged_df = pd.merge(merged_df, earliest_rows[['stop_id', 'shape_id']], on=['stop_id', 'shape_id'], suffixes=('_filtered', '_earliest'))
# Function to convert time to timedelta, handling values greater than 24 hours
def convert_to_timedelta(time_str):
    # Split the time string into hours, minutes, and seconds
    parts = time_str.split(':')
    hours, minutes, seconds = map(int, parts)
    
    # Calculate the total seconds for the time
    total_seconds = hours * 3600 + minutes * 60 + seconds
    
    # Create a timedelta object
    return pd.to_timedelta(total_seconds, unit='s')

df = df2 =headway_df = pd.DataFrame()
# Convert 'departure_time' to timedelta
merged_df['departure_time'] = merged_df['departure_time'].apply(convert_to_timedelta)

model_tod_map = {
    'E8': ['AM', 'MD', 'PM', 'EV', 'NT1', 'NT2'],
    'ABM': ['AM', 'MD', 'PM', 'EV1', 'EV2', 'NT1', 'NT2']
}
tod_list = model_tod_map.get(model, [])  # empty list if model not found

# Get route_name for mathing route_id
def extract_route_from_shape_id(shape_id):
    matches = re.findall(r'shp-(\d+)-', str(shape_id))
    return matches[0] if matches else "unknown"

def get_route_name_logic(shape_id, index_df, auth):
    # Check if shape_id is in index_df
    idx_match = index_df[index_df['gtfs_route'] == shape_id]

    if not idx_match.empty:
        # Existing route found in index
        model_route = idx_match['model_route'].iloc[0]
        return f"{model_route}"
    else:
        # New branch logic
        numeric_route = extract_route_from_shape_id(shape_id)
        # Use '_0' as you specified in your snippet
        route_str = '_0'
        return f"{auth}{route_str}{numeric_route}_Branch - {shape_id}"
    
for time in tod_list:    
    # Define user-defined time period
    start_time = convert_to_timedelta(time_dic[time][0])
    end_time = convert_to_timedelta(time_dic[time][1])
    
    # Filter data within the time period
    filtered_df = merged_df[(merged_df['departure_time'] >= start_time) & (merged_df['departure_time'] <= end_time)]      

    if not filtered_df.empty:
        # Sort the DataFrame by stop, and route
        filtered_df = filtered_df.sort_values(['stop_id', 'shape_id', 'direction_id', 'departure_time']).copy()

        most_common_stops = filtered_df.groupby(['shape_id', 'direction_id'])['stop_id'].agg(lambda x: x.value_counts().idxmax())
        filtered_df = filtered_df[filtered_df['stop_id'].isin(most_common_stops)]
        filtered_df['TOD'] = time[:2]    
        df = pd.concat([df, filtered_df], ignore_index=True)
    else:
        # If empty, don't add anything to 'df' for this time period. This results in missing columns/rows that fillna(999) will handle later.
        pass

# E8: Convert trips from NT to EV or AM. Place holder, not applied yet.
'''
if model == 'E8': 
    am_start1 = pd.to_timedelta("02:00:00")
    am_end1   = pd.to_timedelta("06:29:59")

    am_start2 = pd.to_timedelta("26:00:00")
    am_end2   = pd.to_timedelta("30:29:59")

    am_start3 = pd.to_timedelta("1 days 02:00:00")
    am_end3   = pd.to_timedelta("1 days 06:29:59")

    ev_start1 = pd.to_timedelta("22:00:00")
    ev_end1   = pd.to_timedelta("25:59:59")

    ev_start2 = pd.to_timedelta("1 days 00:00:00")
    ev_end2   = pd.to_timedelta("1 days 01:59:59")

    # Only adjust rows where TOD == 'NT'
    mask_nt = df['TOD'] == 'NT'

    # Reassign to AM
    df.loc[mask_nt & (df['departure_time'].between(am_start1, am_end1)), 'TOD'] = 'AM'
    df.loc[mask_nt & (df['departure_time'].between(am_start2, am_end2)), 'TOD'] = 'AM'
    df.loc[mask_nt & (df['departure_time'].between(am_start3, am_end3)), 'TOD'] = 'AM'

    # Reassign to EV (late night or after midnight)
    df.loc[mask_nt & (df['departure_time'].between(ev_start1, ev_end1)), 'TOD'] = 'EV'
    df.loc[mask_nt & (df['departure_time'].between(ev_start2, ev_end2)), 'TOD'] = 'EV'
'''
    
for time in ['AM', 'MD', 'PM', 'EV', 'NT']:    
    filtered_df2 = df[df['TOD'] == time].copy()
    filtered_df2['trip_headsign'] = filtered_df2['trip_headsign'].astype(str)
    filtered_df2['trip_headsign'].fillna(' ', inplace=True)
    # Subtract one day for values greater than '1 days 03:00:00'
    if model == 'E8' and time == 'EV':
        threshold = pd.to_timedelta('6:30:00')
        filtered_df2.loc[filtered_df2['departure_time'] < threshold, 'departure_time'] += pd.to_timedelta('1 days')
    if model == 'ABM'and time == 'EV':
        threshold = pd.to_timedelta('03:00:00')
        filtered_df2.loc[filtered_df2['departure_time'] < threshold, 'departure_time'] += pd.to_timedelta('1 days')

    if not pd.api.types.is_timedelta64_dtype(filtered_df2['departure_time']): # Ensure departure_time is timedelta
        filtered_df2['departure_time'] = pd.to_timedelta(filtered_df2['departure_time'], errors='coerce')

    # Compute time_diff safely
    filtered_df2 = filtered_df2[filtered_df2['stop_sequence'] == 1] # some stop_times.txt contains stop_sequence other than 1 which should be skipped (e.g. LET)
    # Sort (critical for consistent grouping)
    if time == 'AM': # remove "1 days"
        filtered_df2['departure_time'] = filtered_df2['departure_time'] - pd.to_timedelta(filtered_df2['departure_time'].dt.days, unit='d')
    filtered_df2 = filtered_df2.sort_values(['stop_id', 'shape_id', 'direction_id', 'departure_time']).copy()       
    time_diff = filtered_df2.groupby(['shape_id', 'direction_id', 'stop_id'])['departure_time'].diff()

    # Convert to minutes; if not timedelta, fallback to NaN
    if pd.api.types.is_timedelta64_dtype(time_diff):
        filtered_df2['time_diff'] = time_diff.dt.total_seconds() / 60
    else:
        filtered_df2['time_diff'] = time_diff  # fallback: leave as is (probably NaT or float)

    filtered_df2 = filtered_df2[['arrival_time','departure_time','stop_id','stop_sequence','timepoint',
                               'shape_id','route_id','trip_headsign','direction_id','time_diff', 'TOD']]   
    df2 = pd.concat([df2, filtered_df2], ignore_index=True)
    df2['time_diff'] = df2['time_diff'].replace('', np.nan)  # Handle empty strings
    #df2['TOD'] = df2['TOD'].where(df2['time_diff'].notna())  # Remove TOD if time_diff is NaN

    if not filtered_df2.empty:
        grouped = filtered_df2.groupby(['shape_id', 'route_id', 'trip_headsign', 'direction_id', 'stop_id'])
        if model == 'E8':
            result_df = grouped['departure_time'].agg(  # option2 only
                lambda x: duration[time] if len(x) <= 1 
                        else (x.max() - x.min()) / (len(x) - 1)
            ).reset_index(name=time)
        elif model == 'ABM':
            result_df = grouped['departure_time'].agg(  # if AM, MD, PM - option2, if EV, NT - option 1
                lambda x: duration[time] if len(x) <= 1 
                        else (duration[time] / len(x) if time in ['EV', 'NT'] 
                                else (x.max() - x.min()) / (len(x) - 1))
            ).reset_index(name=time)

        result_df.columns = ['shape_id', 'route_id', 'trip_headsign', 'direction_id', 'stop_id', time]
        result_df[time] = result_df[time].apply(lambda x: int(x.total_seconds() / 60 + 0.5) if isinstance(x, pd.Timedelta)\
                                                          else 0)
        result_df = result_df.sort_values(by=[time], ascending=[False])
        result_df = result_df.drop_duplicates(subset=['shape_id', 'route_id', 'trip_headsign', 'direction_id'], keep='first')
        
        if time == 'AM':
            headway_df = pd.concat([headway_df, result_df[['shape_id', 'route_id', 'trip_headsign', 'direction_id', time]]], ignore_index=True)
        else:
            headway_df = headway_df.merge(result_df[['shape_id', 'route_id', 'trip_headsign', 'direction_id', time]], \
                                        on = ['shape_id', 'route_id', 'trip_headsign', 'direction_id'], how = 'outer')

headway_df.fillna(999, inplace=True)   

# Remove NT for E8
if model == 'E8': 
    if auth == 'LET':
        df2 = df2[(df2['TOD'] != 'NT') & df2['TOD'] != 'EV'].copy()
        if 'EV' in headway_df.columns:
            headway_df = headway_df.drop(columns=['EV'])    
        if 'NT' in headway_df.columns:
            headway_df = headway_df.drop(columns=['NT'])
    else:
        df2 = df2[df2['TOD'] != 'NT'].copy()
        if 'NT' in headway_df.columns:
            headway_df = headway_df.drop(columns=['NT'])

if 'NT' in headway_df.columns:
    headway_df = headway_df.drop(headway_df[(headway_df['AM'] + headway_df['MD'] + headway_df['PM'] + headway_df['EV'] + headway_df['NT']) == 4995.0].index) # 999*5
elif 'EV' in headway_df.columns:
    headway_df = headway_df.drop(headway_df[(headway_df['AM'] + headway_df['MD'] + headway_df['PM'] + headway_df['EV']) == 3996.0].index)
else:
    headway_df = headway_df.drop(headway_df[(headway_df['AM'] + headway_df['MD'] + headway_df['PM']) == 2997.0].index)

# Add service date to headway_df
headway_df.insert(0, 'service_date', service_date)
# Add route name
df2['route_name'] = df2['shape_id'].apply(lambda x: get_route_name_logic(x, index_df, auth))
df2 = df2[['departure_time','stop_id','route_name','shape_id','trip_headsign','direction_id','time_diff','TOD']]
headway_df['route_name'] = headway_df['shape_id'].apply(lambda x: get_route_name_logic(x, index_df, auth))
if auth == 'LET':   
    headway_df = headway_df[['service_date','route_name','shape_id','route_id','trip_headsign','direction_id','AM','MD','PM','NT']]
else:
    headway_df = headway_df[['service_date','route_name','shape_id','route_id','trip_headsign','direction_id','AM','MD','PM','EV','NT']]

# Sort df2
df2 = df2.sort_values(by=['route_name', 'departure_time']).copy()

df2.to_csv(_join(GTFS_location, sub_path, auth + '_' + model + '_GTFS_time_diff.csv'), index=False)
headway_df.to_csv(_join(GTFS_location, sub_path, auth + '_' + model + '_GTFS_headway.csv'), index=False)

# Plot

In [ ]:
street_map = gpd.read_file(_join(GTFS_location, 'All Street Shape File\All_Street.shp'))
# street_map = gpd.read_file(_join(map_location, 'All Street Shape File\All_Street.shp'))
map_format = 'model_network' # input 4: model_network vs all_street

In [ ]:
# Plot TRAU itineraries with background street map
shapes_df2 = pd.read_csv(_join(GTFS_location, sub_path, 'shapes.txt'))
shapes_df = trips_df.merge(shapes_df2, on='shape_id', how='left')
shapes_df = shapes_df[shapes_df['service_id'] == service_id] 
shapes_df = shapes_df.drop_duplicates(subset=['shape_id', 'shape_pt_sequence']) # removal candidate

if auth == 'LET':
    shapes_df['shape_id'] = shapes_df['shape_id'].astype(str)
    index_df['gtfs_route'] = index_df['gtfs_route'].astype(str)

shapes_df = shapes_df.merge(index_df, left_on='shape_id', right_on='gtfs_route', how='left')
shapes_df['model_route'].fillna('new route', inplace=True)
shapes_df['route_id'] = shapes_df['route_id'].astype(str) # need comment for UMI
time_columns = ['shape_id', 'AM', 'MD', 'PM', 'EV', 'NT']
existing_columns = [col for col in time_columns if col in headway_df.columns]

if existing_columns:
    if auth == 'LET':
            headway_df['shape_id'] = headway_df['shape_id'].astype(str)
            # It's also safe to ensure shapes_df is string here just in case
            shapes_df['shape_id'] = shapes_df['shape_id'].astype(str)

    shapes_df = shapes_df.merge(headway_df[existing_columns], on='shape_id', how='left')
    existing_columns.remove('shape_id')
    shapes_df['freq'] = shapes_df[existing_columns].astype(str).apply(','.join, axis=1)

In [ ]:
# Create a PDF file to store the plots WITHOUT stops

# --- Helper Functions ---
def natural_keys(text):
    return [int(c) if c.isdigit() else c for c in re.split(r'(\d+)', str(text))]

def get_sort_key(shape_id, index_df):
    numeric_route = extract_route_from_shape_id(shape_id)
    try:
        route_num = int(numeric_route)
    except:
        route_num = 999 
    idx_match = index_df[index_df['gtfs_route'] == shape_id]
    if not idx_match.empty:
        return (route_num, 0, idx_match.index[0])
    else:
        return (route_num, 1, natural_keys(shape_id))

# --- Prepare Routes ---
all_shape_ids = shapes_df['shape_id'].unique().tolist()
unique_routes = sorted(all_shape_ids, key=lambda x: get_sort_key(x, index_df))

pdf_path = _join(GTFS_location, sub_path, auth + '_' + model + '_GTFS_plots_' + map_format + '.pdf')
pdf_pages = pdf_backend.PdfPages(pdf_path)

# Use try/finally to prevent "Damaged File" errors
try:
    for route in unique_routes:
        route_data_df = shapes_df[shapes_df['shape_id'] == route].copy()
        if route_data_df.empty:
            continue

        shape_id = route_data_df['shape_id'].iloc[0]
        
        # Check if route is in index_df
        idx_match = index_df[index_df['gtfs_route'] == shape_id]
        
        if not idx_match.empty:
            # Existing route
            model_route = idx_match['model_route'].iloc[0]
            # FIXED: changed iloc[1] to iloc[0] to prevent crash on short data
            route_id = route_data_df['route_id'].iloc[0] 
            route_name = f'{model_route} - {shape_id}'
        else:
            # New branch
            numeric_route = extract_route_from_shape_id(shape_id)
            #route_str = '_0' if auth == 'AATA' else '_'
            route_str = '_0'
            route_name = f'{auth}{route_str}{numeric_route}_Branch - {shape_id}'
            route_id = f'{auth}{route_str}{numeric_route}'

        # Frequency logic
        raw_freq = str(route_data_df['freq'].iloc[0])
        if raw_freq != 'nan':
            route_freq = ','.join([v.replace('.0', '') for v in raw_freq.split(',')])
        else:
            route_freq = "N/A"
            
        if model == 'E8':  
            if auth == 'LET':   
                route_freq = route_freq + ',999,999'
            else:
                route_freq = route_freq + ',999'

        # Plotting
        route_geometry = LineString(zip(route_data_df['shape_pt_lon'], route_data_df['shape_pt_lat']))
        route_bbox = route_geometry.bounds
        cropped_map = street_map.cx[route_bbox[0]:route_bbox[2], route_bbox[1]:route_bbox[3]]
        
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(route_data_df['shape_pt_lon'], route_data_df['shape_pt_lat'], color='blue', zorder=2)

        # Labels
        ax.text(route_data_df['shape_pt_lon'].iloc[0], route_data_df['shape_pt_lat'].iloc[0], 'A', color='red', fontweight='bold', zorder=3)
        ax.text(route_data_df['shape_pt_lon'].iloc[-1], route_data_df['shape_pt_lat'].iloc[-1], 'B', color='green', fontweight='bold', zorder=3)

        if not cropped_map.empty:
            cropped_map.plot(ax=ax, color='lightgrey', linewidth=0.3, zorder=1)    
        
        ax.set_title(f'{route_id, route_name, route_freq, model}')
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')
        ax.grid(False)
        
        pdf_pages.savefig(fig)
        plt.close(fig) # Releases memory

finally:
    # This ALWAYS runs, even if the loop above crashes. 
    # This prevents the "Damaged File" error in Adobe.
    pdf_pages.close()
    print("PDF closed successfully.")